# train-eval-mode-branch — faded example 3: Collect outputs under train and eval mode from Sequential

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `train-eval-mode-branch`. Running the beacon reports progress on the `PyTorch: train/eval mode` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: train/eval mode` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`train-eval-mode-branch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "train-eval-mode-branch"
DD_SUBTOPIC = "PyTorch: train/eval mode"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Calling `model.train()` or `model.eval()` propagates the flag recursively to every submodule in a `Sequential` container. All modules inside — including Dropout, BatchNorm, and plain Linear layers — update their `training` attribute together. You can check any individual submodule's flag to confirm the propagation.

## Faded exercise 3

Implement `faded3_sequential_mode_outputs(model, x, n_reps)`. Call `model.train()`. Compute `train_outs = [model(x) for _ in range(n_reps)]`. Then call `model.eval()`. Compute `eval_outs = [model(x) for _ in range(n_reps)]`. Return a dict with `train_outs`, `eval_outs`, and `eval_all_equal` (bool: whether all outputs in `eval_outs` are allclose to each other).

**Fill in:** The model.train() call that puts the Sequential into training mode before collecting train_outs.

In [ ]:
import torch as t
import torch.nn as nn

def faded3_sequential_mode_outputs(model, x, n_reps):
    model.train()
    train_outs = [model(x) for _ in range(n_reps)]
    model.eval()
    eval_outs = [model(x) for _ in range(n_reps)]
    eval_all_equal = all(
        t.allclose(eval_outs[0], eval_outs[i]) for i in range(1, n_reps)
    )
    return {'train_outs': train_outs, 'eval_outs': eval_outs, 'eval_all_equal': eval_all_equal}


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(9)
    model = nn.Sequential(nn.Linear(5, 5), nn.Dropout(p=0.5), nn.Linear(5, 2))
    x = t.randn(4, 5)
    result = faded3_sequential_mode_outputs(model, x, n_reps=10)
    assert len(result['train_outs']) == 10
    assert len(result['eval_outs']) == 10
    # Eval mode: deterministic
    assert result['eval_all_equal'] == True
    # Train mode: stochastic (with high probability, not all equal)
    train_all_eq = all(
        t.allclose(result['train_outs'][0], result['train_outs'][i])
        for i in range(1, 10)
    )
    assert not train_all_eq, 'train mode should be stochastic with dropout'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def faded3_sequential_mode_outputs(model, x, n_reps):
    model.train()
    train_outs = [model(x) for _ in range(n_reps)]
    model.eval()
    eval_outs = [model(x) for _ in range(n_reps)]
    eval_all_equal = all(
        t.allclose(eval_outs[0], eval_outs[i]) for i in range(1, n_reps)
    )
    return {'train_outs': train_outs, 'eval_outs': eval_outs, 'eval_all_equal': eval_all_equal}
```
</details>